# The first approach to any EDA process is to understand the table content and structure

### Target tables:

- SalesOrderHeader: financial focuses on revenue, orders, dates and customers
- SalesOrderDetails: financial focuses in line items, products, quantities, prices
- Product: financial focuses on Products, prices and costs
- Employee: human focuses on employees, managers, hire dates, salaries
- Customer: human focuses on Customers, territories, demographics

### these tables focuses on extracting potentional `human-centered` and `financial` insights for the business direct and indirect benefit

In [34]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text

load_dotenv()

db_user = os.getenv('DB_USER')
db_host = os.getenv('DB_HOST')
encoded_password = quote_plus(os.getenv('DB_PASSWORD'))
encoded_name = quote_plus(os.getenv('DB_NAME'))

connection_string = f'mysql+pymysql://{db_user}:{encoded_password}@{db_host}/{encoded_name}'
engine = create_engine(connection_string)

with engine.connect() as connection:
    query = 'SELECT * FROM `aw2022-sales-salesorderheader`'
    df = pd.read_sql_query(query, connection)

## A robust query design must:

- save sensitive variables in .env
- encode strings before reaching the DB connection string
- include a parameter dictionary as a guardrail against SQL injection
- align with business goals as to not overengineer the task and overcomplicate it

### for our case, we wouldn't include the third option as we are handling manual EDA and not an automated pipeline for a live service

In [35]:
df.head()

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,...,CreditCardID,CreditCardApprovalCode,CurrencyRateID,SubTotal,TaxAmt,Freight,TotalDue,Comment,rowguid,ModifiedDate
0,43659,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43659,PO522145787,10-4020-000676,...,16281.0,105041Vi84182,NaN,20565.6206,1971.5149,616.0984,23153.2339,None,{79B65321-39CA-4115-9CBA-8FE0903E12E6},2011-06-07
1,43660,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43660,PO18850127500,10-4020-000117,...,5618.0,115213Vi29411,NaN,1294.2529,124.2483,38.8276,1457.3288,None,{738DC42D-D03B-48A1-9822-F95A67EA7389},2011-06-07
2,43661,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43661,PO18473189620,10-4020-000442,...,1346.0,85274Vi6854,4.0,32726.4786,3153.7696,985.5530,36865.8012,None,{D91B9131-18A4-4A11-BC3A-90B6F53E9D74},2011-06-07
3,43662,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43662,PO18444174044,10-4020-000227,...,10456.0,125295Vi53935,4.0,28832.5289,2775.1646,867.2389,32474.9324,None,{4A1ECFC0-CC3A-4740-B028-1C50BB48711C},2011-06-07
4,43663,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43663,PO18009186470,10-4020-000510,...,4322.0,45303Vi22691,NaN,419.4589,40.2681,12.5838,472.3108,None,{9B1E7A40-6AE0-4AD3-811C-A64951857C4B},2011-06-07


In [36]:
df.shape

(31465, 26)

In [37]:
df.describe()

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,CustomerID,SalesPersonID,TerritoryID,BillToaddressID,ShipToAddressID,ShipMethodID,CreditCardID,CurrencyRateID,SubTotal,TaxAmt,Freight,TotalDue,ModifiedDate
count,31465.000000,31465.000000,31465,31465,31465,31465.0,31465.000000,3806.000000,31465.000000,31465.000000,31465.000000,31465.000000,30334.000000,13976.000000,31465.000000,31465.000000,31465.000000,31465.000000,31465
mean,59391.000000,8.000953,2013-08-21 11:43:49.321468,2013-09-02 11:44:14.034641,2013-08-28 11:44:14.034641,5.0,20170.175687,280.607987,6.090768,18263.154426,18249.192563,1.483839,9684.100448,9191.499571,3491.065673,323.755743,101.173693,3915.995109,2013-08-28 11:44:14.034641
min,43659.000000,8.000000,2011-05-31 00:00:00,2011-06-12 00:00:00,2011-06-07 00:00:00,5.0,11000.000000,274.000000,1.000000,405.000000,9.000000,1.000000,1.000000,2.000000,1.374000,0.109900,0.034400,1.518300,2011-06-07 00:00:00
25%,51525.000000,8.000000,2013-06-20 00:00:00,2013-07-02 00:00:00,2013-06-27 00:00:00,5.0,14432.000000,277.000000,4.000000,14080.000000,14063.000000,1.000000,4894.250000,8510.000000,56.970000,4.557600,1.424300,62.951900,2013-06-27 00:00:00
50%,59391.000000,8.000000,2013-11-03 00:00:00,2013-11-15 00:00:00,2013-11-10 00:00:00,5.0,19452.000000,279.000000,6.000000,19449.000000,19438.000000,1.000000,9719.500000,10074.000000,782.990000,62.639200,19.574800,865.204000,2013-11-10 00:00:00
75%,67257.000000,8.000000,2014-02-28 00:00:00,2014-03-13 00:00:00,2014-03-08 00:00:00,5.0,25994.000000,284.000000,9.000000,24678.000000,24672.000000,1.000000,14510.750000,11282.000000,2366.960000,189.597600,59.249300,2615.490800,2014-03-08 00:00:00
max,75123.000000,9.000000,2014-06-30 00:00:00,2014-07-12 00:00:00,2014-07-07 00:00:00,5.0,30118.000000,290.000000,10.000000,29883.000000,29883.000000,5.000000,19237.000000,12431.000000,163930.394300,17948.518600,5608.912100,187487.825000,2014-07-07 00:00:00
std,9083.307446,0.030864,NaN,NaN,NaN,0.0,6261.728960,4.846965,2.958119,8210.069158,8218.429263,1.304343,5566.299591,2945.170095,11093.452536,1085.054180,339.079427,12515.462713,NaN


In [61]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 31465 entries, 0 to 31464
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   SalesOrderID            31465 non-null  int64         
 1   RevisionNumber          31465 non-null  int64         
 2   OrderDate               31465 non-null  datetime64[us]
 3   DueDate                 31465 non-null  datetime64[us]
 4   ShipDate                31465 non-null  datetime64[us]
 5   Status                  31465 non-null  int64         
 6   OnlineOrderFlag         31465 non-null  str           
 7   SalesOrderNumber        31465 non-null  str           
 8   PurchaseOrderNumber     3806 non-null   str           
 9   AccountNumber           31465 non-null  str           
 10  CustomerID              31465 non-null  int64         
 11  SalesPersonID           3806 non-null   float64       
 12  TerritoryID             31465 non-null  int64         
 1

## it appears the next columns contain non-null values:

- PurchaseOrderNumber
- SalesPersonID
- CreditCardID
- CreditCardApprovalCode
- CurrencyRateID

In [51]:
def nan_count(column):
    ''' a function that returns the count of nan values for different columns'''
    nan = 0
    non_nan = 0

    for i in df[column].isna():
        if i == True:
            nan += 1
        elif i == False:
            non_nan += 1

    return nan, non_nan

for i in df.columns:
    nan, non_nan = nan_count(i)
    total = nan + non_nan

    if total == 0 or nan == 0:
        continue

    ratio = nan / total

    print(f'{i}, nan_ratio: {ratio*100:.2f}%')

PurchaseOrderNumber, nan_ratio: 87.90%
SalesPersonID, nan_ratio: 87.90%
CreditCardID, nan_ratio: 3.59%
CreditCardApprovalCode, nan_ratio: 3.59%
CurrencyRateID, nan_ratio: 55.58%
Comment, nan_ratio: 100.00%


## Generated insights from nan_ratios:
- 88% missing values in `SalePersonID` and `PurchaseOrderNumber` allows to define an intial assumption that 88% of sales transactions were online orders
- 4% in `CreditCardID` and `CreditCardApprovalCode` means that 4% of orders were paid using cash or other paying mediums whereby a massive ratio of 96% of transactions were paid using a credit card
    - it interestingly align with our first insight whom assumes 88% of sales were online orders, we will calculate the `OnlineFlagRatio` to confirm this assumption
- for the `CurrencyRateID` column, it appears that almost 56% of transactions happened in the local market (domestic), whereby almost 44% happened in internal or foreign markets

In [58]:
true, false = df['OnlineOrderFlag'].value_counts()

true_ratio = true / (true + false)
print(f'online_orders_ratio: {true_ratio*100:.2f}%')

online_orders_ratio: 87.90%


- Online Orders Ratio: 87.90%
- Conclusion: Our previous assumption is true; the vast majority of transactions were online orders

### in Explatory Data Analysis some columns must be `dropped` as they hold 0 to low value in the process
##### we might try to save some NaN records if the ratio of missing values are relatively low and can be propagated
- columns to be removed:
    - Comment: an empty column
    - rowguid: Global Unique Identifier, doesn't add value to the EDA step
    - ModifiedData: data entry's date of modification, doesn't add value
- columns to be propagated:
    - None

In [62]:
columns_to_remove = ['Comment', 'rowguid', 'ModifiedDate']
refined_df = df.drop(columns=columns_to_remove)

refined_df.head()

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,...,BillToaddressID,ShipToAddressID,ShipMethodID,CreditCardID,CreditCardApprovalCode,CurrencyRateID,SubTotal,TaxAmt,Freight,TotalDue
0,43659,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43659,PO522145787,10-4020-000676,...,985,985,5,16281.0,105041Vi84182,NaN,20565.6206,1971.5149,616.0984,23153.2339
1,43660,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43660,PO18850127500,10-4020-000117,...,921,921,5,5618.0,115213Vi29411,NaN,1294.2529,124.2483,38.8276,1457.3288
2,43661,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43661,PO18473189620,10-4020-000442,...,517,517,5,1346.0,85274Vi6854,4.0,32726.4786,3153.7696,985.5530,36865.8012
3,43662,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43662,PO18444174044,10-4020-000227,...,482,482,5,10456.0,125295Vi53935,4.0,28832.5289,2775.1646,867.2389,32474.9324
4,43663,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43663,PO18009186470,10-4020-000510,...,1073,1073,5,4322.0,45303Vi22691,NaN,419.4589,40.2681,12.5838,472.3108


In [75]:
# calculating the taxratio before freight cost
refined_df['TaxRatio%'] = np.round(refined_df['TaxAmt'] / (refined_df['SubTotal'] + refined_df['TaxAmt'])*100, 2)

refined_df

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,...,ShipToAddressID,ShipMethodID,CreditCardID,CreditCardApprovalCode,CurrencyRateID,SubTotal,TaxAmt,Freight,TotalDue,TaxRatio%
0,43659,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43659,PO522145787,10-4020-000676,...,985,5,16281.0,105041Vi84182,NaN,20565.6206,1971.5149,616.0984,23153.2339,8.75
1,43660,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43660,PO18850127500,10-4020-000117,...,921,5,5618.0,115213Vi29411,NaN,1294.2529,124.2483,38.8276,1457.3288,8.76
2,43661,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43661,PO18473189620,10-4020-000442,...,517,5,1346.0,85274Vi6854,4.0,32726.4786,3153.7696,985.5530,36865.8012,8.79
3,43662,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43662,PO18444174044,10-4020-000227,...,482,5,10456.0,125295Vi53935,4.0,28832.5289,2775.1646,867.2389,32474.9324,8.78
4,43663,8,2011-05-31,2011-06-12,2011-06-07,5,FALSE,SO43663,PO18009186470,10-4020-000510,...,1073,5,4322.0,45303Vi22691,NaN,419.4589,40.2681,12.5838,472.3108,8.76
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31460,75119,8,2014-06-30,2014-07-12,2014-07-07,5,TRUE,SO75119,NaN,10-4030-011981,...,17649,1,6761.0,429826Vi35166,NaN,42.2800,3.3824,1.0570,46.7194,7.41
31461,75120,8,2014-06-30,2014-07-12,2014-07-07,5,TRUE,SO75120,NaN,10-4030-018749,...,28374,1,8925.0,929849Vi46003,NaN,84.9600,6.7968,2.1240,93.8808,7.41
31462,75121,8,2014-06-30,2014-07-12,2014-07-07,5,TRUE,SO75121,NaN,10-4030-015251,...,26553,1,14220.0,529864Vi73738,NaN,74.9800,5.9984,1.8745,82.8529,7.41
31463,75122,8,2014-06-30,2014-07-12,2014-07-07,5,TRUE,SO75122,NaN,10-4030-015868,...,14616,1,18719.0,330022Vi97312,NaN,30.9700,2.4776,0.7743,34.2219,7.41


In [76]:
refined_df['SubTotal'].describe()

count     31465.000000
mean       3491.065673
std       11093.452536
min           1.374000
25%          56.970000
50%         782.990000
75%        2366.960000
max      163930.394300
Name: SubTotal, dtype: float64

### the creation of `TaxRatio%` urge the need for more geographic data, as it appears although some subtotals are low average yet with higher tax than much larger subtotal values
- further analysis in shipment interval can be considered as indirect cause of tax increase
- further spatial data needed to evaluate a more accurate description of tax differentiation
- perhaps low subtotal have a higher tax rate on average, or different market static taxes regardless of such, we will proceed to confirm this assumption

### we will start with the first hypothesis as to confirm if low subtotal shipments have a greater taxratio than bigger subtotal shipments, followed by freight cost impact on taxratio